# Lesson 1 — Your First Legal-AI Tool: Summarize a Contract Clause with Claude

**The hook:** in the next ~30 minutes you'll turn a dense contract clause into a clear, plain-English summary — using about five lines of Python and one call to Claude.

> **By the end you'll have shipped:** a working `summarize_clause()` function — the first slice of our **Matter Intelligence** capstone tool.

|  |  |
|---|---|
| **Module** | M2 · Building with Claude |
| **Prerequisites** | None — this is a from-scratch starting point |
| **Est. time** | 30-40 minutes |
| **Capstone slice** | "Read & summarize one document" |
| **Difficulty** | Core (with optional `Go Deeper 🔧`) |

*This notebook runs end-to-end with **no API key and no internet** — it uses a built-in simulator. Add a key later for real Claude output. Nothing here is legal advice; always have a lawyer review AI output.*

## What you'll be able to do

By the end of this lesson, you'll be able to:

1. Explain what "calling an LLM" actually means, in plain terms.
2. Send Claude a prompt from Python and read its answer back.
3. Use a **system prompt** to set Claude's role and rules (like briefing a junior associate).
4. Summarize a contract clause into plain English on demand.
5. Package your work into a reusable function you'll build on next lesson.

## Why it matters ⚖️

Lawyers read a *lot* of boilerplate. A limitation-of-liability clause, a governing-law provision, an indemnity — each dense, each important, each slow to skim for the hundredth time.

An LLM can produce a **first-pass plain-English summary** in seconds, so a person spends their time *judging* the clause instead of *decoding* it. That's the whole thesis of this course: use LLMs to remove the drudgery and keep the human in charge of the judgment.

We'll start with the smallest useful version of that — summarizing **one clause** — and grow it into a real tool over the coming lessons.

In [ ]:
# --- Lesson 1 setup: run this cell first ------------------------------------
# Safe to re-run. Needs NO API key and NO internet to work.

import os, json, textwrap

# (Optional) Make the Anthropic SDK available for when you add a real key later.
# If you're offline, this quietly does nothing and the lesson still runs.
try:
    import anthropic  # noqa: F401
except ImportError:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=False)
    except Exception:
        pass

# --- Model configuration (one central place; IDs change over time) -----------
# Live list: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_SMART    = "claude-opus-4-8"    # hardest reasoning
MODEL_BALANCED = "claude-sonnet-5"    # great default for most tasks
MODEL_FAST     = "claude-haiku-4-5"   # cheap & fast

# --- MOCK mode: auto-on when there's no API key ------------------------------
# MOCK True  -> answers are simulated locally (no network, no cost)
# MOCK False -> real calls to Claude (needs ANTHROPIC_API_KEY)
MOCK = not os.environ.get("ANTHROPIC_API_KEY")

# --- A synthetic sample clause (never use real client data in a lesson) ------
SAMPLE_CLAUSES = {
    "limitation_of_liability": (
        "Limitation of Liability. Except for breaches of confidentiality or "
        "indemnification obligations, in no event shall either party's aggregate "
        "liability arising out of or related to this Agreement exceed the total "
        "fees paid by Customer to Provider in the twelve (12) months preceding the "
        "event giving rise to the claim. In no event shall either party be liable "
        "for any indirect, incidental, consequential, special, or punitive damages, "
        "including lost profits, even if advised of the possibility of such damages."
    ),
    "governing_law": (
        "Governing Law; Venue. This Agreement shall be governed by and construed in "
        "accordance with the laws of the State of Delaware, without regard to its "
        "conflict of laws principles. The parties consent to the exclusive "
        "jurisdiction of the state and federal courts located in Wilmington, Delaware."
    ),
}

print("Setup complete.")
print("MOCK mode:", MOCK, "(simulated responses)" if MOCK else "(real Claude calls)")
if MOCK:
    print("-> Add an ANTHROPIC_API_KEY to your environment for real output. The lesson works either way.")

In [ ]:
# --- ask_claude(): our one helper for talking to Claude ----------------------
# In MOCK mode it returns a realistic *simulated* answer so the whole notebook
# runs offline. With a key, it calls the real API and returns the text.

def _simulated_answer(system, prompt):
    text = ((system or "") + " " + prompt).lower()

    if "hello" in text or "introduce yourself" in text:
        return ("Hello! I'm Claude. In this course I'll be your tireless junior "
                "associate - summarizing, extracting, and drafting on request, "
                "always for a human to review.")

    # Guess the clause topic so the simulation feels responsive.
    if "governing law" in text or "delaware" in text or "jurisdiction" in text:
        topic = "which state's law governs and where disputes are heard"
    elif "liability" in text or "damages" in text or "indemn" in text:
        topic = "how much each side can be forced to pay if things go wrong"
    else:
        topic = "the parties' key obligations"

    if "json" in text or "extract" in text:
        return json.dumps({
            "clause_type": "limitation_of_liability",
            "liability_cap": "12 months of fees paid",
            "excluded_damages": ["indirect", "consequential", "punitive", "lost profits"],
            "carve_outs": ["confidentiality breaches", "indemnification"],
        }, indent=2)

    if "risk" in text:
        return ("Key risks to flag: (1) the cap is tied to fees paid, which may be "
                "small relative to potential harm; (2) consequential damages are "
                "fully excluded; (3) confidentiality and indemnity sit OUTSIDE the "
                "cap, creating uncapped exposure there. A lawyer should confirm.")

    # Default: a plain-English summary.
    return ("[Simulated summary] In plain English, this clause is about " + topic + ". "
            "It limits each party's total liability to the fees paid in the prior 12 "
            "months and excludes indirect or consequential damages such as lost "
            "profits. Two things are carved out and NOT capped: confidentiality "
            "breaches and indemnification. (Turn off MOCK mode for a real, "
            "clause-specific answer.)")

def ask_claude(prompt, system=None, model=MODEL_BALANCED, max_tokens=400):
    # Simulated path - no network, no key, no cost.
    if MOCK:
        return _simulated_answer(system, prompt)

    # Real path - talk to Claude.
    from anthropic import Anthropic
    client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    kwargs = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if system:
        kwargs["system"] = system
    message = client.messages.create(**kwargs)
    return message.content[0].text

# Quick smoke test:
print(ask_claude("Say hello in one short sentence."))

## Concept 1 — What is "calling an LLM"?

Think of Claude as an extraordinarily well-read **junior associate**. You hand it a written **instruction** (a *prompt*), and it hands back a written **work product** (a *response*).

That's the whole loop:

> **prompt in → response out.**

There's no elaborate setup — it's one function call. Everything else in this course is about writing *better instructions* and *wiring the responses into tools*.

The setup cell already ran `ask_claude(...)` once for a smoke test. Let's use it for something real.

In [ ]:
clause = SAMPLE_CLAUSES["limitation_of_liability"]
print("THE CLAUSE:\n")
print(textwrap.fill(clause, width=88))

print("\n\nCLAUDE'S SUMMARY:\n")
summary = ask_claude("Summarize this contract clause in plain English:\n\n" + clause)
print(textwrap.fill(summary, width=88))

**What just happened:** you sent Claude a one-line instruction plus the clause, and got back a plain-English summary. Useful — but we gave it no guidance on *who it is* or *how to answer*. That's the job of the **system prompt**.

## Concept 2 — The system prompt: briefing your associate ⚖️

Before you hand a junior associate a task, you brief them: *"You summarize contract clauses for busy lawyers. Be concise and neutral, and flag anything unusual. Never give legal advice."*

The **system prompt** is exactly that standing briefing. It sets role, tone, and rules **once**, and applies to everything that follows. Let's give Claude a proper brief.

In [ ]:
SYSTEM_BRIEF = (
    "You summarize contract clauses for busy lawyers. "
    "Write 2-3 sentences in plain English. Be neutral and precise. "
    "Call out any carve-outs, caps, or unusual terms. "
    "You are not a lawyer and do not give legal advice."
)

summary = ask_claude(
    "Summarize this clause for a lawyer skimming a contract:\n\n" + clause,
    system=SYSTEM_BRIEF,
)
print(textwrap.fill(summary, width=88))

**What just happened:** same clause, but now the *briefing* shapes the answer — its length, tone, and what it flags. You'll reuse one good system prompt across thousands of documents.

> ### Go Deeper 🔧 — two dials worth knowing
> - **`max_tokens`** caps how long the answer can be (its length budget). Too low truncates the answer; higher just allows more room.
> - **`temperature`** (0–1) controls randomness. For legal summarization you usually want it **low** (~0–0.3) so answers are consistent and faithful. We'll set it explicitly in a later lesson.

> ### Common pitfalls ⚠️
> - **Don't paste real client data** into a lesson (or any external tool without approval). We use synthetic clauses on purpose.
> - **Always verify.** An LLM can sound confident and still be wrong. The human signs off, not the model.

## Your turn 🛠️

Time to drive. Edit the `# TODO` lines and run the cells. Solutions are at the very bottom if you get stuck.

In [ ]:
# Exercise 1 - Summarize a DIFFERENT clause.
# TODO: change the key from "limitation_of_liability" to "governing_law".
my_clause = SAMPLE_CLAUSES["limitation_of_liability"]   # <-- edit this line

print(textwrap.fill(ask_claude(
    "Summarize this clause in plain English:\n\n" + my_clause,
    system=SYSTEM_BRIEF,
), width=88))

In [ ]:
# Exercise 2 - Change the AUDIENCE.
# TODO: rewrite the brief so the summary is aimed at a NON-lawyer business client
#       (short, no legalese, explain what it means for them in practice).
my_brief = "TODO: write your own system brief here"

print(textwrap.fill(ask_claude(
    "Summarize this clause:\n\n" + SAMPLE_CLAUSES["limitation_of_liability"],
    system=my_brief,
), width=88))

In [ ]:
# Exercise 3 (stretch) - Ask a different QUESTION, not a summary.
# TODO: write a prompt that asks Claude to list the top risks in the clause
#       for the Customer specifically. (Tip: include the word "risk".)
my_prompt = "TODO: write your prompt here (mention the word 'risk')"

print(textwrap.fill(ask_claude(
    my_prompt + "\n\n" + SAMPLE_CLAUSES["limitation_of_liability"],
    system=SYSTEM_BRIEF,
), width=88))

## Build the capstone slice 🧱

Everything so far becomes **one reusable function** — the first brick of **Matter Intelligence**. Next lesson we'll have it return *structured* data; today it returns a clean summary, and can aim that summary at either a lawyer or a business client.

In [ ]:
def summarize_clause(clause_text, audience="lawyer"):
    # audience: "lawyer" or "client" - shapes the briefing.
    brief = (
        "You summarize contract clauses. Write 2-3 plain-English sentences, "
        "flag caps/carve-outs/unusual terms, and never give legal advice."
    )
    if audience == "client":
        brief += " Write for a non-lawyer business reader; avoid legalese."
    return ask_claude(
        "Summarize this contract clause:\n\n" + clause_text,
        system=brief,
        model=MODEL_BALANCED,
    )

# Ship it - try both audiences on the governing-law clause:
for who in ("lawyer", "client"):
    print("=== Summary for a " + who + " ===")
    print(textwrap.fill(summarize_clause(SAMPLE_CLAUSES["governing_law"], audience=who), width=88))
    print()

## Recap — what you shipped ✅

You built a working `summarize_clause()` function. Along the way you learned:

- **prompt in → response out** is the whole LLM loop.
- The **system prompt** briefs Claude on role, tone, and rules — and is reused across every call.
- The same clause can be summarized for **different audiences** just by changing the brief.
- A notebook can run **offline in MOCK mode**, then produce real output the moment you add an API key.

**Artifact shipped:** `summarize_clause()` — the "read & summarize one document" slice of Matter Intelligence.

## Check your understanding 🤔

<details>
<summary>1. In one sentence, what does a "system prompt" do?</summary>

It gives Claude a standing briefing — role, tone, and rules — that applies to every message, like briefing a junior associate before handing over work.
</details>

<details>
<summary>2. Why does this notebook still run with no API key?</summary>

Because `MOCK` turns on automatically when no key is present, and `ask_claude()` returns a realistic **simulated** answer instead of calling the network.
</details>

<details>
<summary>3. Where should your Claude API key live — and where should it NOT?</summary>

In an **environment variable** (e.g., `ANTHROPIC_API_KEY`), never hardcoded in a cell or committed to a repo.
</details>

## Next lesson → structured output 🎯

Right now Claude hands us a paragraph. Next lesson we'll ask it for **structured JSON** — pulling the parties, the liability cap, the excluded damages, and the carve-outs into clean fields we can store in a database. That's the bridge from "nice summary" to "data we can build on."

**Prep:** nothing to install. Bring an API key if you have one (optional).

## Reference & glossary 📚

| Term | Plain meaning | Legal analogy |
|---|---|---|
| **LLM** | A model that turns text instructions into text answers | A very well-read junior associate |
| **Prompt** | The instruction you send | The assignment memo |
| **Response / completion** | What the model sends back | The associate's draft |
| **System prompt** | Standing role and rules for the model | The briefing before the assignment |
| **Token** | The unit of text length models count | Words / word-pieces on the billable clock |
| **`max_tokens`** | Length budget for the answer | "Keep it to a page" |
| **MOCK mode** | Local simulated answers, no network | A practice run with a stand-in |

**Used in this lesson:** `ask_claude()`, `summarize_clause()`, `client.messages.create(...)`.

**Docs:** Models — https://platform.claude.com/docs/en/about-claude/models/overview · Messages API — https://platform.claude.com/docs/en/api/messages

## Solutions (peek only after trying) 🔑

<details>
<summary>Show solutions</summary>

**Exercise 1** — change the key:

`my_clause = SAMPLE_CLAUSES["governing_law"]`

**Exercise 2** — a client-facing brief:

`my_brief = "You explain contract clauses to non-lawyer business clients. Use 2-3 short sentences, no legalese, and say what it means for them in practice. You are not a lawyer and give no legal advice."`

**Exercise 3** — a risk prompt:

`my_prompt = "List the top risks this clause creates for the Customer, as short bullet points."`
</details>